# 2.1 Intro and API basics: requests, multi-turn chat, system prompts

Step 2, lesson 1 of [Building with the Claude API](https://anthropic.skilljar.com/claude-with-the-anthropic-api) (Anthropic Academy).
Exam domains: **4 Prompt Engineering & Structured Output** (system prompts) and **5 Context Management & Reliability** (the API is stateless, so *you* manage the conversation).

What this notebook covers, in the course's order:

1. Getting an API key and keeping it out of the code
2. Making a first request, and reading the response
3. Multi-turn conversations: the API remembers nothing, so we resend the history
4. A small chat loop
5. System prompts: telling Claude *how* to behave
6. Temperature, and what changed on current models

> **Model used here:** `claude-sonnet-5-5`. The course notebooks name an older model; swap the `model` variable to try another one.

## 1. API key and client

You create a key in the [Claude Console](https://platform.claude.com) under **Settings → API keys**. A key is a password for your account's credits, so:

- it lives in a `.env` file at the repo root (`ANTHROPIC_API_KEY=sk-ant-...`), which is in `.gitignore`;
- the code never contains it and never prints it;
- `load_dotenv()` copies it into the environment, and `Anthropic()` reads it from there.

In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()          # finds the .env file in this folder or a parent folder
client = Anthropic()   # reads ANTHROPIC_API_KEY from the environment
model = "claude-sonnet-5-5"

## 2. A first request

Every call goes to one endpoint, **Messages** (`client.messages.create`). Three arguments are required:

| Argument | What it is |
|---|---|
| `model` | which Claude to use |
| `max_tokens` | a hard cap on how much Claude may write (a safety limit, not a target length) |
| `messages` | the conversation so far: a list of `{"role": "user" or "assistant", "content": ...}` |

In [2]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[{"role": "user", "content": "What is quantum computing? Answer in one sentence."}],
)
message

Message(id='msg_011Ch5UKVUVZqZFFBudjcg7s', container=None, content=[TextBlock(citations=None, text='Quantum computing is a type of computing that uses quantum-mechanical phenomena like superposition and entanglement, through units called qubits, to solve certain complex problems much faster than classical computers can.', type='text')], diagnostics=None, model='claude-sonnet-5-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=25, output_tokens=65, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard', speed='standard'))

The answer is a `Message` object. The parts worth knowing:

- `content`: a **list of blocks**. Text is in blocks of type `text`; later lessons add `tool_use` and `thinking` blocks.
- `stop_reason`: why Claude stopped (`end_turn` = finished, `max_tokens` = cut off by your cap, `tool_use` = wants to call a tool).
- `usage`: input and output tokens, which is what you pay for.

> **Changed since the course.** The course reads the answer with `message.content[0].text`. On current models the first block can be a `thinking` block (adaptive thinking is on by default), so pick the first block whose type is `text`.

In [3]:
def text_of(message):
    """Return the text of the first text block in a response."""
    return next(block.text for block in message.content if block.type == "text")

print([block.type for block in message.content], message.stop_reason)
print(message.usage.input_tokens, "input tokens,", message.usage.output_tokens, "output tokens")
print(text_of(message))

['text'] end_turn
25 input tokens, 65 output tokens
Quantum computing is a type of computing that uses quantum-mechanical phenomena like superposition and entanglement, through units called qubits, to solve certain complex problems much faster than classical computers can.


## 3. Multi-turn conversations

The API is **stateless**: it keeps nothing between calls. To have a conversation, you keep the list of messages yourself and send *all of it* every time, alternating `user` and `assistant`.

The course builds three small helpers for this, and the rest of the course uses them:

In [4]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, max_tokens=1000):
    params = {"model": model, "max_tokens": max_tokens, "messages": messages}
    if system:
        params["system"] = system
    message = client.messages.create(**params)
    return text_of(message)

In [5]:
messages = []

add_user_message(messages, "Define quantum computing in one sentence.")
answer = chat(messages)
add_assistant_message(messages, answer)
print("Claude:", answer, "\n")

# A follow-up that only makes sense with the history
add_user_message(messages, "Write another sentence that builds on that one.")
answer = chat(messages)
add_assistant_message(messages, answer)
print("Claude:", answer)

Claude: Quantum computing is a type of computation that uses quantum-mechanical phenomena such as superposition, entanglement, and interference to process information in qubits, allowing it to solve certain problems far more efficiently than classical computers. 



Claude: Unlike classical bits, which are strictly 0 or 1, qubits can exist in a combination of both states at once, which lets quantum algorithms such as Shor's (for factoring large numbers) and Grover's (for searching unsorted data) tackle tasks that would take classical machines impractically long.


Without the history, the second question has no meaning. Same question, sent alone:

In [6]:
print(chat([{"role": "user", "content": "Write another sentence that builds on that one."}]))

I don't see a previous sentence in our conversation, so I'm missing what to build on. Could you paste it here?

If you'd like, I can also start with a sentence of my own and then add to it. For example:

*The old lighthouse had stood empty for decades.* → *Yet every night, a faint light still flickered in its highest window.*

Just send me your sentence, and tell me the tone or genre you're going for if you have one in mind.


**Why this matters for the exam (Domain 5).** Because you resend the whole history on every call, a long conversation costs more on every turn and eventually fills the context window. Everything in Domain 5 (summarising old turns, trimming tool results, keeping "case facts" in a fixed block) is a way of managing this list that *you* own.

## 4. A small chat loop

The course's exercise is a loop: read a line from the user, add it, ask Claude, add the answer, repeat. In Jupyter you would use `input()`; here a fixed list of lines stands in for the person typing, so the notebook runs from top to bottom.

In [7]:
messages = []
user_lines = ["Hi! My name is Sam.", "I'm learning the Claude API.", "What's my name, and what am I learning?"]

for line in user_lines:            # interactive version: while True: line = input("> ")
    add_user_message(messages, line)
    answer = chat(messages, max_tokens=300)
    add_assistant_message(messages, answer)
    print("You:   ", line)
    print("Claude:", answer, "\n")

You:    Hi! My name is Sam.
Claude: Hi Sam! Nice to meet you. How can I help you today? 



You:    I'm learning the Claude API.
Claude: That's a great thing to learn! I can help with a few things:

- **Getting started**: setting up your API key, installing the SDK (Python or TypeScript), and making your first request
- **Core concepts**: the Messages API, system prompts, multi-turn conversations, and parameters like `max_tokens` and `temperature`
- **Features**: streaming, tool use (function calling), vision, prompt caching, and structured outputs
- **Best practices**: error handling, rate limits, cost management, and prompt design

For reference, here's a minimal Python example:

```python
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from your environment

message = client.messages.create(
    model="claude-sonnet-4-20250 



You:    What's my name, and what am I learning?
Claude: Your name is Sam, and you're learning the Claude API.

My last message got cut off in the middle of the code example. Here's the complete version:

```python
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from your environment

message = client.messages.create(
     



## 5. System prompts

A **system prompt** sets the role and rules for the whole conversation. It is a separate `system` argument, not a message. The course's example is a maths tutor who should guide, not give away answers.

In [8]:
system = """You are a patient math tutor.
Do not directly answer a student's question.
Guide them to a solution step by step, with one hint at a time."""

question = [{"role": "user", "content": "How do I solve 5x + 2 = 3 for x?"}]

print("--- without a system prompt ---")
print(chat(question, max_tokens=400))
print("\n--- with the tutor system prompt ---")
print(chat(question, system=system, max_tokens=400))

--- without a system prompt ---


**Step 1: Subtract 2 from both sides.**

5x + 2 − 2 = 3 − 2
5x = 1

**Step 2: Divide both sides by 5.**

x = 1/5

**Answer:** x = 1/5 (or 0.2)

**Check:** 5(1/5) + 2 = 1 + 2 = 3 ✓

--- with the tutor system prompt ---


Good question! This is a linear equation, and the goal is to get **x** alone on one side of the equal sign.

Right now, x is being multiplied by 5, and then 2 is added to it. To undo those operations, we work backward, starting with the addition.

**Hint 1:** What could you do to *both sides* of the equation to get rid of the "+ 2" on the left?

Try it and tell me what you get.


**Exam angle (Domain 4).** A system prompt is the right first tool for *tone, role and style*. But it is guidance: Claude follows it very reliably, not with certainty. When a rule **must always hold** (never refund above a limit, always verify identity first), the guide's answer is to enforce it in code, for example a check before a tool runs, rather than adding more emphatic wording to the prompt.

## 6. Temperature, and what changed

In the course, `temperature` (0 to 1) controls randomness: low for predictable answers, high for creative ones.

> **Changed since the course.** On current models (Sonnet 5.5, Opus 5.5, Haiku 5.5) a non-default `temperature`, `top_p` or `top_k` is rejected, and from API version `2026-10-21` they are rejected on every model. The current knob is **effort** (`output_config={"effort": "low" | "medium" | "high" | ...}`), which sets how much the model thinks before answering.

The current Python SDK (2.x) has already dropped the argument, so the call fails in Python before anything is sent:

In [9]:
try:
    client.messages.create(
        model=model,
        max_tokens=100,
        temperature=0.0,
        messages=[{"role": "user", "content": "Say hi."}],
    )
except TypeError as e:
    print("Python SDK:", e)

Python SDK: Messages.create() got an unexpected keyword argument 'temperature'


Forcing it into the request body with `extra_body` shows what the API itself says:

In [10]:
import anthropic

try:
    client.messages.create(
        model=model,
        max_tokens=100,
        messages=[{"role": "user", "content": "Say hi."}],
        extra_body={"temperature": 0.0},
    )
except anthropic.BadRequestError as e:
    print("API 400:", e.message)

API 400: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'temperature: Extra inputs are not permitted'}, 'request_id': 'req_011Ch5ULtmUmxrorbCnD5Y13'}


In [11]:
# The current way to trade depth for speed and cost: effort
quick = client.messages.create(
    model=model,
    max_tokens=300,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "Give me a one-line name idea for a coffee shop."}],
)
print(text_of(quick), "|", quick.usage.output_tokens, "output tokens")

**"Bean There, Brewed That"** | 16 output tokens


## Recap

- One endpoint, `messages.create`, with `model`, `max_tokens` and `messages`.
- Read text from the blocks of type `text`; check `stop_reason` and `usage`.
- The API is stateless: resend the whole conversation each time. You own the history (Domain 5).
- The system prompt sets role and rules; rules that must always hold are enforced in code (Domain 4).
- `temperature` is gone on current models; `effort` is the knob now.

Next: [2.2 Streaming and structured data](02-streaming-and-structured-data.ipynb).